In [73]:
import requests
import pandas as pd
import numpy as np
from io import StringIO
import yfinance as yf

# Task 1

In [2]:
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

In [3]:
response = requests.get(url, headers=headers)

In [11]:
response.raise_for_status()

In [13]:
data = pd.read_html(StringIO(response.text))

In [20]:
df = pd.DataFrame(data[0])

In [22]:
df_cut = df[['Security', 'Symbol', 'Date added']].copy()

In [24]:
df_cut['Year added'] = pd.to_datetime(df_cut['Date added'], errors='coerce').dt.year

In [74]:
df_cut.loc[df_cut['Year added'] >= 2020, 'Year added'].value_counts().sort_values(ascending=False)

,count
Year added,
2025,18
2024,16
2023,15
2022,15
2026,13
2021,10
2020,10


# Task 2

In [33]:
indices = {
    'United States': '^GSPC',
    'China': '000001.SS',
    'Hong Kong': '^HSI',
    'Australia': '^AXJO',
    'India': '^NSEI',
    'Canada': '^GSPTSE',
    'Germany': '^GDAXI',
    'United Kingdom': '^FTSE',
    'Japan': '^N225',
    'Mexico': '^MXX',
    'Brazil': '^BVSP'
}

In [34]:
start_date = '2026-01-01'
end_date = '2026-08-21'

In [35]:
results = []

In [36]:
for country, ticker in indices.items():
    data = yf.download(ticker, start=start_date, end=end_date, progress=False)
    if data.empty:
        print(f"No data for {country} ({ticker})")
        continue
    first_close = data['Close'].iloc[0]
    last_close = data['Close'].iloc[-1]
    ytd_return = (last_close / first_close - 1) * 100
    results.append({'Country': country, 'Ticker': ticker, 'YTD Return (%)': round(float(ytd_return), 2)})

/tmp/ipykernel_1509/1838511292.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipykernel_1509/1838511292.py:9: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  results.append({'Country': country, 'Ticker': ticker, 'YTD Return (%)': round(float(ytd_return), 2)})
/tmp/ipykernel_1509/1838511292.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipykernel_1509/1838511292.py:9: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  results.append({'Country': country, 'Ticker': ticker, 'YTD Return (%)': round(float(ytd_return), 2)})
/tmp/ipykernel_1509/1838511292.py:2: FutureWarni

In [37]:
ytd_df = pd.DataFrame(results).sort_values('YTD Return (%)', ascending=False).reset_index(drop=True)

In [38]:
ytd_df

,Country,Ticker,YTD Return (%)
0,Japan,^N225,27.75
1,Canada,^GSPTSE,14.06
2,United States,^GSPC,11.41
3,United Kingdom,^FTSE,8.01
4,Germany,^GDAXI,5.88
5,Brazil,^BVSP,4.60
6,Australia,^AXJO,4.08
7,Mexico,^MXX,0.32
8,Hong Kong,^HSI,-2.43
9,China,000001.SS,-2.97


In [39]:
sp500_return = ytd_df.loc[ytd_df['Country'] == 'United States', 'YTD Return (%)'].values[0]

In [40]:
beat_sp500 = ytd_df[(ytd_df['Country'] != 'United States') & (ytd_df['YTD Return (%)'] > sp500_return)]

In [42]:
beat_sp500

,Country,Ticker,YTD Return (%)
0,Japan,^N225,27.75
1,Canada,^GSPTSE,14.06


In [43]:
print(f"Number of indexes (out of 10) beating the S&P 500: {len(beat_sp500)}")

Number of indexes (out of 10) beating the S&P 500: 2


# Task 3

In [75]:
sp500 = yf.download('^GSPC', start='1950-01-01', end='2026-09-12', progress=False)

/tmp/ipykernel_1509/3842734196.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  sp500 = yf.download('^GSPC', start='1950-01-01', end='2026-09-12', progress=False)


In [76]:
sp500 = sp500[['Close']].copy()

In [77]:
sp500.columns = ['Close']
sp500 = sp500.dropna()

In [78]:
sp500['Running_Max'] = sp500['Close'].cummax()
sp500['Is_ATH'] = sp500['Close'] == sp500['Running_Max']

In [79]:
ath_dates = sp500[sp500['Is_ATH']].index
ath_prices = sp500.loc[ath_dates, 'Close']

In [82]:
corrections = []

In [83]:
for i in range(len(ath_dates) - 1):
    start_date = ath_dates[i]
    end_date = ath_dates[i + 1]

    window = sp500.loc[start_date:end_date, 'Close']

    if len(window) <= 1:
        continue

    peak_price = window.iloc[0]
    trough_price = window.min()
    trough_date = window.idxmin()

    drawdown_pct = (peak_price - trough_price) / peak_price * 100
    duration_days = (trough_date - start_date).days

    corrections.append({
        'Peak_Date': start_date,
        'Peak_Price': peak_price,
        'Trough_Date': trough_date,
        'Trough_Price': trough_price,
        'Drawdown_%': drawdown_pct,
        'Duration_Days': duration_days
    })

In [84]:
corrections_df = pd.DataFrame(corrections)

In [85]:
significant_corrections = corrections_df[corrections_df['Drawdown_%'] >= 5].copy()
significant_corrections = significant_corrections.sort_values('Drawdown_%', ascending=False).reset_index(drop=True)

In [86]:
duration_percentiles = significant_corrections['Duration_Days'].quantile([0.25, 0.5, 0.75])
drawdown_percentiles = significant_corrections['Drawdown_%'].quantile([0.25, 0.5, 0.75])

In [87]:
print(f"\nMedian correction drawdown: {duration_percentiles[0.5]:.1f} days, {drawdown_percentiles[0.5]:.1f}% drawdown")


Median correction drawdown: 40.5 days, 8.0% drawdown


# Task 4

In [155]:
ticker = 'AMZN'
ticker_obj = yf.Ticker(ticker)
result = ticker_obj.get_earnings_dates()

In [156]:
earnings = result.reset_index()

In [157]:
earnings = earnings.rename(columns={"Earnings Date": "earnings_date"})

In [158]:
earnings["earnings_date"] = pd.to_datetime(earnings["earnings_date"])

In [159]:
earnings["earnings_date"] = earnings["earnings_date"].dt.tz_localize(None)

In [160]:
earnings = earnings.dropna(subset=["Surprise(%)"])

In [161]:
earnings = earnings.sort_values("earnings_date")

In [162]:
prices = yf.download(ticker, period="max", auto_adjust=False)

[*********************100%***********************]  1 of 1 completed


In [163]:
if isinstance(prices.columns, pd.MultiIndex):
    prices.columns = prices.columns.get_level_values(0)

In [164]:
prices = prices.reset_index()

In [165]:
prices["Date"] = pd.to_datetime(prices["Date"])

In [166]:
if prices["Date"].dt.tz is not None:
    prices["Date"] = prices["Date"].dt.tz_localize(None)

In [167]:
prices = prices.sort_values("Date").reset_index(drop=True)

In [168]:
prices["close_day1"] = prices["Close"]
prices["close_day3"] = prices["Close"].shift(-2)

In [169]:
prices["return_2d"] = (prices["close_day3"] / prices["close_day1"] - 1)

In [170]:
prices["earnings_date"] = prices["Date"].shift(-1)

In [171]:
returns = prices[["Date", "earnings_date", "return_2d"]].copy()

In [172]:
returns = returns.dropna(subset=["return_2d"])

In [173]:
earnings["earnings_day"] = earnings["earnings_date"].dt.normalize()
returns["earnings_day"] = returns["earnings_date"].dt.normalize()

In [174]:
merged = returns.merge(earnings[["earnings_day","Reported EPS","EPS Estimate","Surprise(%)"]], on="earnings_day", how="inner")

In [175]:
positive_surprises = merged[merged["Surprise(%)"] > 0].copy()

In [176]:
median_2d_return = positive_surprises["return_2d"].median()

In [177]:
print(f"Median 2-day return: "f"{median_2d_return:.2%}")

Median 2-day return: 0.35%
